# Scenario: Investigating Truncated Clinical Summaries

In [8]:
import pandas as pd
import sqlite3
# creating Dataset representing clinical intake logs
clinical_notes = {
    "record_id": [201, 202, 203, 204, 205],
    "patient_id": ["P-88", "P-89", "P-90", "P-91", "P-92"],   # Imagine the system has a hidden 50-character limit
    "intake_summary": 
    ["Patient exhibits mild hypertension and slight cough.",   # 51 chars (Truncated!)
     "History of chronic asthma; currently stable.",           # 41 chars
     "Patient complains of acute abdominal pain radiating t",  # 53 chars (Truncated!)
     "Routine checkup. No acute distress noted.",              # 42 chars
     "Allergic to penicillin; requires alternative antibiot"   # 54 chars (Truncated!)
    ]
}
# adding dataset into the DataFrame
df_clinical_notes = pd.DataFrame(clinical_notes)
# creating sql to save the DataFrame into temp memory
connt = sqlite3.connect(":memory:")
df_clinical_notes.to_sql("clinicalNotes", connt, index = False, if_exists = "replace")
# creating function to run the query 
def run_query(query):
    return pd.read_sql_query(query, connt)
print("********************************** Text Length Audit Database is ready! **************")



********************************** Text Length Audit Database is ready! **************


# Measuring Text Lengths

In [14]:
# query to review all data
all_data = "SELECT * FROM clinicalNotes"
print("*********************************** all data to review ************** ")
display(run_query(all_data))
print()
"""query that uses the LENGTH() function to calculate the character count of each intake_summary. 
Return the record_id, the text itself, and its length as character_count"""
data_length = """
SELECT 
    record_id,
    intake_summary,
    LENGTH (intake_summary) AS charactor_count
FROM clinicalNotes
"""
print("************************** data length ******************")
display(run_query(data_length))

*********************************** all data to review ************** 


,record_id,patient_id,intake_summary
0,201,P-88,Patient exhibits mild hypertension and slight ...
1,202,P-89,History of chronic asthma; currently stable.
2,203,P-90,Patient complains of acute abdominal pain radi...
3,204,P-91,Routine checkup. No acute distress noted.
4,205,P-92,Allergic to penicillin; requires alternative a...



************************** data length ******************


,record_id,intake_summary,charactor_count
0,201,Patient exhibits mild hypertension and slight ...,52
1,202,History of chronic asthma; currently stable.,44
2,203,Patient complains of acute abdominal pain radi...,53
3,204,Routine checkup. No acute distress noted.,41
4,205,Allergic to penicillin; requires alternative a...,53


# Detecting the Limit Bottleneck

In [18]:
# query to isolate only the records where the intake_summary length is greater than or equal to 50 characters.
limit_bottleneck = """
SELECT
    record_id,
    intake_summary,
    LENGTH(intake_summary) AS charactor_count
FROM clinicalNotes
WHERE LENGTH(intake_summary) >= 50;
"""
print("********************************** limit bottleneck **************")
display(run_query(limit_bottleneck))

********************************** limit bottleneck **************


,record_id,intake_summary,charactor_count
